In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [4]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Kamtera/ParsiGoo", 
    repo_type="dataset", local_dir="./ParsiGoo")

Fetching 5 files: 100%|██████████| 5/5 [00:03<00:00,  1.63it/s]


'/home/ubuntu/ParsiGoo'

In [7]:
!ls ParsiGoo/text_corpus

sampel_text.csv


In [10]:
data_dir = 'ParsiGoo'
data_dir = os.path.join(data_dir, "datasets")
print("| >  data_dir =",data_dir)
meta_files = []
speaker_names = os.listdir(data_dir)
root_path = ""
print("| >  listdir =",os.listdir(data_dir))
for speaker_name in os.listdir(data_dir):
    # if not os.path.isdir(os.path.join(data_dir, speaker_name)):
        # continue
    root_path = os.path.join(data_dir, speaker_name)
    meta_files.append(os.path.join(root_path, "metadata.csv"))

| >  data_dir = ParsiGoo/datasets
| >  listdir = ['ariana_Male2', 'edge_Farid', 'ariana_Female1', 'ariana_Male1', 'edge_Dilara', 'moujeze_Female1']


In [22]:
txt_files = meta_files
rows = []
id=-1
for ind,txt_file in enumerate(txt_files):
  with open(txt_file, "r", encoding="utf-8") as ttf:
    for i, line in enumerate(ttf):
        cols = line.split("|")
        wav_file = cols[1].strip()
        text = cols[0].strip()
        wav_file = os.path.join(root_path, "wavs", wav_file)
        id+=1
        rows.append({"text": text, "audio_file": wav_file, "speaker_name": speaker_names[ind], "root_path": root_path})

In [59]:
rows[0]

{'text': 'حتی اندام\u200cهای جنسی شما خون بیشتری دریافت می\u200cکنند، و توان جنسی شما افزایش می\u200cیابد.',
 'audio_file': 'ParsiGoo/datasets/moujeze_Female1/wavs/5821-z.wav',
 'speaker_name': 'ariana_Male2',
 'root_path': 'ParsiGoo/datasets/moujeze_Female1'}

In [43]:
!ls ParsiGoo/datasets/moujeze_Female1/wavs/19-63-fa.wav

ParsiGoo/datasets/moujeze_Female1/wavs/19-63-fa.wav


In [65]:
def loop(rows):

    rows, _ = rows

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'

    data = []
    for row in tqdm(rows):
        try:
            f = row['audio_file']
            base = f.split('/')[0] + '_audio'
            f_new = f.replace('/', '-').replace('.parquet', '').replace('.wav', '')
            os.makedirs(base, exist_ok=True)
            t = row['text'].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            audio_np, sr = sf.read(f)
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{row['speaker_name']}"
            })
        except Exception as e:
            pass
        
    return data

In [66]:
data = loop((rows, 0))

100%|██████████| 2701/2701 [00:14<00:00, 183.93it/s]  


In [67]:
audio_files = [d['audio_filename'] for d in data]

with open('ParsiGoo-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [68]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'ParsiGoo_audio/ParsiGoo-datasets-moujeze_Female1-wavs-16-104-fa-ima2ia7g9qo_224.mp3',
 'text': 'آنان که به آیات خداوند ایمان نمی آورند',
 'speaker': 'ParsiGoo_audio_moujeze_Female1'}

In [69]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'ParsiGoo')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 1792.44ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 21.9kB / 21.9kB,   ???B/s  
Processing Files (1 / 1): 100%|██████████| 21.9kB / 21.9kB,  0.00B/s  
New Data Upload: 100%|██████████| 21.9kB / 21.9kB,  0.00B/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  2.60 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/43c25591546721e6661965bb117f369eb4caddb2', commit_message='Upload dataset', commit_description='', oid='43c25591546721e6661965bb117f369eb4caddb2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [72]:
# !zip -rq ParsiGoo_audio.zip ParsiGoo_audio

In [73]:
# !hf upload malaysia-ai/Multilingual-TTS ParsiGoo_audio.zip --repo-type=dataset